In [1]:
%load_ext autoreload
%autoreload 2
    
import os
import sys
import pandas as pd
import xarray as xr
import numpy as np
from dask.distributed import Client
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from functools import partial

from cdrmip_extremes import (
    file_search, load_raw, amoc, processing, utils, io_disk
)
from cdrmip_extremes.utils import _map_leaves, reduce_member_tree
from cdrmip_extremes.configs import DATA_DIR, RAW_DIR, PROC_DIR, CDR_MODELS as models, CDR_EXPTS as expts

In [2]:
client = Client(memory_limit=None,threads_per_worker=1,n_workers=28)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/43371/status,
Dashboard: /proxy/43371/status,Workers: 28
Total threads: 28,Total memory: 0 B
Status: running,Using processes: True
Comm: tcp://127.0.0.1:38919,Workers: 0
Dashboard: /proxy/43371/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:33725,Total threads: 1
Dashboard: /proxy/36615/status,Memory: 0 B
Nanny: tcp://127.0.0.1:35007,


In [3]:
msftmz_dir = RAW_DIR / 'msftmz-msftyz'
msftmz = defaultdict(lambda: defaultdict(dict))

for model in models:
    for expt in expts:
        ds = xr.open_mfdataset(
            str(msftmz_dir / expt / model / '*.nc'),
            use_cftime=True,
            drop_variables=['lev_bnds'],
            chunks={'time': 120},
        )
        msftmz[model][ds.attrs['experiment_id']][ds.attrs['variant_label']] = ds

In [4]:
msftmz

defaultdict(<function __main__.<lambda>()>,
            {'ACCESS-ESM1-5': defaultdict(dict,
                         {'1pctCO2': {'r1i1p1f1': <xarray.Dataset> Size: 333MB
                           Dimensions:    (time: 1800, bnds: 2, lat: 300, basin: 3, lev: 50)
                           Coordinates:
                             * time       (time) object 14kB 0101-01-16 12:00:00 ... 0250-12-16 12:00:00
                             * lat        (lat) float64 2kB -77.75 -77.51 -77.26 -77.01 ... 89.1 89.55 90.0
                               sector     (basin) |S21 63B dask.array<chunksize=(3,), meta=np.ndarray>
                             * lev        (lev) float64 400B 5.0 15.0 25.0 ... 5.499e+03 5.831e+03
                           Dimensions without coordinates: bnds, basin
                           Data variables:
                               time_bnds  (time, bnds) object 29kB dask.array<chunksize=(120, 2), meta=np.ndarray>
                               lat_bnds   (time, lat

In [23]:
branch_date_overrides = {} 

expt = '1pctCO2-cdr'
concat = defaultdict(lambda: defaultdict(dict))
for model, expts in msftmz.items():
    parents = expts.get('1pctCO2', {})
    for member, ds in expts.get(expt, {}).items():
        parent = parents.get(ds.attrs.get('parent_variant_label', member))
        if parent is None:
            print(f'No 1pctCO2 parent for {model} {member}, skipping')
            continue
        concat[model][expt][member] = processing.concatenate_branches(
            ds, parent, branch_date=branch_date_overrides.get(model), realign_years = (0,140)
        ).assign_attrs(experiment_id=expt)

concat = utils.defaultdict_to_dict(concat)

Concatenating via realign_years=(0, 140) (parent relabeled to end 0139-12-16 12:00:00, child relabeled to start 0140-01-16 12:00:00)
Concatenating via realign_years=(0, 140) (parent relabeled to end 0139-12-16 12:00:00, child relabeled to start 0140-01-16 12:00:00)
Concatenating via realign_years=(0, 140) (parent relabeled to end 0140-01-15 12:00:00, child relabeled to start 0140-01-15 12:45:00)
Concatenating via realign_years=(0, 140) (parent relabeled to end 0139-12-16 12:00:00, child relabeled to start 0140-01-16 12:00:00)
Concatenating via realign_years=(0, 140) (parent relabeled to end 0139-12-16 12:00:00, child relabeled to start 0140-01-16 12:00:00)
Concatenating via realign_years=(0, 140) (parent relabeled to end 0139-12-16 12:00:00, child relabeled to start 0140-01-16 12:00:00)
Concatenating via realign_years=(0, 140) (parent relabeled to end 0139-12-16 12:00:00, child relabeled to start 0140-01-16 12:00:00)
Concatenating via realign_years=(0, 140) (parent relabeled to end 013

In [24]:
concat

{'ACCESS-ESM1-5': {'1pctCO2-cdr': {'r1i1p1f1': <xarray.Dataset> Size: 2GB
   Dimensions:    (time: 10800, bnds: 2, lat: 300, basin: 3, lev: 50)
   Coordinates:
     * time       (time) object 86kB 0000-01-16 12:00:00 ... 0899-12-16 12:00:00
     * lat        (lat) float64 2kB -77.75 -77.51 -77.26 -77.01 ... 89.1 89.55 90.0
       sector     (basin) |S21 63B b'atlantic_arctic_ocean' ... b'global_ocean'
     * lev        (lev) float64 400B 5.0 15.0 25.0 ... 5.499e+03 5.831e+03
   Dimensions without coordinates: bnds, basin
   Data variables:
       time_bnds  (time, bnds) object 173kB dask.array<chunksize=(120, 2), meta=np.ndarray>
       lat_bnds   (time, lat, bnds) float64 52MB dask.array<chunksize=(120, 300, 2), meta=np.ndarray>
       msftmz     (time, basin, lev, lat) float32 2GB dask.array<chunksize=(120, 3, 50, 300), meta=np.ndarray>
   Attributes: (12/46)
       Conventions:            CF-1.7 CMIP-6.2
       activity_id:            CMIP
       branch_method:          standard
   

In [25]:
amoc_dict = _map_leaves(concat, amoc.calc_amoc)
amoc_annual = _map_leaves(amoc_dict, lambda da: da.groupby('time.year').mean().load())

In [32]:
pi_amoc = {
    model: {'piControl': {mem: amoc.calc_amoc(ds).load()
                          for mem, ds in expts['piControl'].items()}}
    for model, expts in msftmz.items() if 'piControl' in expts
}

In [33]:
amoc_anom = defaultdict(lambda: defaultdict(dict))
for model, expts in amoc_annual.items():
    for expt, members in expts.items():
        for mem, da in members.items():
            pi_mem = concat[model][expt][mem].attrs.get('parent_variant_label', mem)
            ref = pi_amoc.get(model, {}).get('piControl', {}).get(pi_mem)
            if ref is not None:
                ref = ref.mean('time')
            if ref is None:
                print(f'No piControl for {model} {pi_mem}, skipping')
                continue
            amoc_anom[model][expt][mem] = da - ref
amoc_anom = utils.defaultdict_to_dict(amoc_anom)

In [34]:
amoc_anom

{'ACCESS-ESM1-5': {'1pctCO2-cdr': {'r1i1p1f1': <xarray.DataArray 'amoc' (year: 900)> Size: 7kB
   array([ 0.19989014,  0.21795082, -1.37286186, -1.13478088, -1.85503578,
          -0.66853142, -0.07315445, -0.440588  , -1.50523949, -0.21608353,
           0.19053078,  0.7798214 , -0.35715103, -1.07798386, -1.76242638,
          -0.36612129,  0.42064285,  0.40785599, -1.3014698 , -1.71615791,
          -1.62618446, -1.35326958, -0.07902145, -0.40120316, -0.71892738,
          -1.47475624, -0.62645912, -2.1305809 , -1.61903381, -1.2684536 ,
          -1.37048912, -1.60256767, -2.81235123, -3.7164526 , -1.75051689,
           0.11954689, -1.31884575, -1.79567909, -1.74818611, -3.06071472,
          -3.28540325, -3.6070528 , -1.29687881, -1.98837662, -2.50546741,
          -2.63802719, -2.16811943, -1.88540268, -3.81906509, -3.80315113,
          -3.63016891, -2.03524971, -2.27029419, -2.00854874, -2.1860323 ,
          -2.44437695, -3.19934177, -4.09732723, -2.37200642, -1.96238327,
     

In [29]:
io_disk.save_tree(
    amoc_anom,
    PROC_DIR,
    var='amoc',
    activity_id='CDRMIP',
    category='anomaly'
)

Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/amoc/ACCESS-ESM1-5/1pctCO2-cdr/ACCESS-ESM1-5_1pctCO2-cdr_r1i1p1f1_amoc_anomaly.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/amoc/CanESM5/1pctCO2-cdr/CanESM5_1pctCO2-cdr_r1i1p2f1_amoc_anomaly.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/amoc/CESM2/1pctCO2-cdr/CESM2_1pctCO2-cdr_r1i1p1f1_amoc_anomaly.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/amoc/CNRM-ESM2-1/1pctCO2-cdr/CNRM-ESM2-1_1pctCO2-cdr_r1i1p1f2_amoc_anomaly.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/amoc/GFDL-ESM4/1pctCO2-cdr/GFDL-ESM4_1pctCO2-cdr_r1i1p1f1_amoc_anomaly.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/amoc/MIROC-ES2L/1pctCO2-cdr/MIROC-ES2L_1pctCO2-cdr_r1i1p1f2_amoc_anomaly.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/amoc/NorESM2-LM/1pctCO2-cdr/NorESM2-LM_1pctCO2-cdr

In [35]:
pi_amoc

{'ACCESS-ESM1-5': {'piControl': {'r1i1p1f1': <xarray.DataArray 'amoc' (time: 12000)> Size: 48kB
   array([19.9175  , 14.946886, 19.347673, ..., 14.010325, 17.061228,
          19.848724], dtype=float32)
   Coordinates:
     * time     (time) object 96kB 0101-01-31 00:00:00 ... 1100-12-31 00:00:00
       sector   |S21 21B b'atlantic_arctic_ocean'
       lat      float64 8B 26.04
   Attributes:
       standard_name:  ocean_meridional_overturning_mass_streamfunction
       long_name:      Ocean Meridional Overturning Mass Streamfunction
       comment:        Overturning mass streamfunction arising from all advectiv...
       units:          kg s-1
       cell_methods:   longitude: sum (comment: basin sum [along zig-zag grid pa...
       history:        2019-11-12T23:12:13Z altered by CMOR: Reordered dimension...}},
 'CanESM5': {'piControl': {'r1i1p2f1': <xarray.DataArray 'amoc' (time: 12612)> Size: 50kB
   array([14.670238, 13.563861, 11.998084, ..., 15.564631, 14.49437 ,
          16.93

In [36]:
io_disk.save_tree(
    pi_amoc,
    PROC_DIR,
    var='amoc',
    activity_id='CDRMIP',
    category=None
)

Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/amoc/ACCESS-ESM1-5/piControl/ACCESS-ESM1-5_piControl_r1i1p1f1_amoc.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/amoc/CanESM5/piControl/CanESM5_piControl_r1i1p2f1_amoc.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/amoc/CESM2/piControl/CESM2_piControl_r1i1p1f1_amoc.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/amoc/CNRM-ESM2-1/piControl/CNRM-ESM2-1_piControl_r1i1p1f2_amoc.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/amoc/GFDL-ESM4/piControl/GFDL-ESM4_piControl_r1i1p1f1_amoc.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/amoc/MIROC-ES2L/piControl/MIROC-ES2L_piControl_r1i1p1f2_amoc.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/processed/CDRMIP/amoc/NorESM2-LM/piControl/NorESM2-LM_piControl_r1i1p1f1_amoc.nc
Saving: /g/data/if69/sc1326/repos/cdrmip_extremes/data/pro